In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer

In [ ]:
# Define the path to the workshop directory
workshop_path = r'C:\Users\shomo\Desktop\workshop'

# Load training dataset from CSV file
train_df = pd.read_csv(os.path.join(workshop_path, 'train.csv'))

# Load testing dataset from CSV file
test_df = pd.read_csv(os.path.join(workshop_path, 'test.csv'))

# Print the shape (rows, columns) of train and test datasets
print("Train:", train_df.shape, "Test:", test_df.shape)

Train: (8693, 14) Test: (4277, 13)


In [ ]:
def feature_engineering(df):
    # Create a copy of input dataframe to avoid modifying original data
    data = df.copy()
    
    # Split PassengerId into Group and Number columns, convert Number to integer
    data[['Group', 'Number']] = data['PassengerId'].str.split('_', expand=True)
    data['Number'] = data['Number'].astype(int)
    
    # Calculate group size and create IsAlone flag (1 if alone, 0 otherwise)
    group_counts = data['Group'].map(data['Group'].value_counts())
    data['IsAlone'] = (group_counts == 1).astype(int)
    data['GroupSize'] = group_counts
    
    # Extract Deck, Side, and Cabin number from Cabin string
    data['Deck'] = data['Cabin'].str[0]
    data['Side'] = data['Cabin'].str.split('/').str[2]
    data['CabinNum'] = data['Cabin'].str.split('/').str[1]
    data['CabinNum'] = pd.to_numeric(data['CabinNum'], errors='coerce')
    
    # Calculate total spending by summing all amenity columns
    spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
    data['TotalSpent'] = data[spend_cols].sum(axis=1)
    
    # Create VIP flag (1 if Deck is A or B, 0 otherwise)
    data['IsVIP'] = (data['Deck'].isin(['A', 'B'])).astype(int)
    
    return data

In [ ]:
# Apply feature engineering to training data
train_processed = feature_engineering(train_df)

# Apply the same feature engineering to test data
test_processed = feature_engineering(test_df)

# Print confirmation message that feature engineering is completed
print("Feature engineering done.")

Feature engineering done.


In [ ]:
# Define lists of numerical and categorical features
num_features = ['Age','RoomService','FoodCourt','ShoppingMall','Spa','VRDeck',
                'GroupSize','IsAlone','TotalSpent','IsVIP','CabinNum','Number']
cat_features = ['HomePlanet','CryoSleep','Destination','Deck','Side']
all_features = num_features + cat_features

# Prepare feature matrix and target variable for training
X_train = train_processed[all_features].copy()
y_train = train_processed['Transported'].astype(int)

# Prepare feature matrix for testing and save passenger IDs for submission
X_test = test_processed[all_features].copy()
passenger_ids = test_processed['PassengerId']

# Print confirmation message that features are prepared
print("Features ready.")

Features ready.


In [ ]:
# Initialize dictionary to store label encoders for each categorical feature
label_encoders = {}

# Encode categorical features using LabelEncoder
for col in cat_features:
    # Combine train and test data to ensure consistent encoding
    combined = pd.concat([X_train[col], X_test[col]], axis=0).astype(str)
    # Create and fit label encoder on combined data
    le = LabelEncoder()
    le.fit(combined)
    # Transform both train and test sets
    X_train[col] = le.transform(X_train[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))
    # Store encoder for later use
    label_encoders[col] = le

# Print confirmation message
print("Encoding done.")

Encoding done.


In [ ]:
# Initialize imputer to fill missing values with median
imputer = SimpleImputer(strategy='median')

# Fit imputer on training data and transform training data
X_train = pd.DataFrame(imputer.fit_transform(X_train), columns=all_features)

# Transform test data using the fitted imputer
X_test = pd.DataFrame(imputer.transform(X_test), columns=all_features)

# Print confirmation message
print("Missing values imputed.")

Missing values imputed.


In [ ]:
# Initialize Random Forest classifier with specified hyperparameters
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, 
                                  min_samples_split=5, min_samples_leaf=2,
                                  random_state=42, n_jobs=-1)

# Train the Random Forest model on the training data
rf_model.fit(X_train, y_train)

# Print confirmation message
print("Model trained.")

Model trained.


In [ ]:
# Perform 5-fold cross-validation and calculate accuracy scores
cv_scores = cross_val_score(rf_model, X_train, y_train, cv=5, scoring='accuracy')

# Print mean and standard deviation of cross-validation accuracy
print(f"CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

CV Accuracy: 0.7920 (+/- 0.0205)


In [ ]:
# Calculate feature importance from the trained Random Forest model
imp = pd.DataFrame({'feature': all_features, 'importance': rf_model.feature_importances_})

# Sort features by importance in descending order
imp = imp.sort_values('importance', ascending=False)

# Display the top 10 most important features
print("Top 10 important features:")
print(imp.head(10))

Top 10 important features:
         feature  importance
8     TotalSpent    0.198536
13     CryoSleep    0.113289
4            Spa    0.099829
1    RoomService    0.092243
2      FoodCourt    0.088519
5         VRDeck    0.085725
3   ShoppingMall    0.062378
10      CabinNum    0.060657
15          Deck    0.049479
0            Age    0.043448


In [ ]:
# Generate predictions on the test set and convert to boolean type
pred = rf_model.predict(X_test).astype(bool)

# Create submission DataFrame with PassengerId and predictions
sub = pd.DataFrame({'PassengerId': passenger_ids, 'Transported': pred})

# Save submission file to the specified path without index
sub.to_csv(os.path.join(workshop_path, 'submission_random_forest.csv'), index=False)

# Print confirmation message with the saved file path
print(f"Saved to {os.path.join(workshop_path, 'submission_random_forest.csv')}")

Saved to C:\Users\shomo\Desktop\workshop\submission_random_forest.csv
